In [1]:
#imports
import os
from pathlib import Path
import random
import csv
from time import sleep
from PIL import Image

In [2]:
import numpy as np
import rasterio
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [3]:
import importnb
with importnb.Notebook():
    from ndvi import get_landsat_ndvi_matrix

In [ ]:
# Set the path to your folder
farms_Folder = Path("./farmlocations") # source of the data
output_Folder = Path("./ndvidata") # where the data will be stored
total = 100

csv_files = farms_Folder.glob("*.csv") #getting all csv files

existing_coords = set()

for csv_path in csv_files: #creating folders
    folder_name = csv_path.stem

    new_folder_path = output_Folder / folder_name

    new_folder_path.mkdir(parents=True, exist_ok=True)
    print(f"Created folder: {new_folder_path}")
    usedRows = []
    with open(csv_path, mode='r') as file:
        reader = csv.reader(file)
        header = next(reader)
        all_rows = list(reader)

        for row in reader:
            row_id,lat,long = row[0].strip(), row[1].strip(), row[2].strip()

            if (lat, long) not in existing_coords:
                usedRows.append(row)
    random.shuffle(all_rows)

    for i in range(total):
        if not all_rows:  # Break early if the CSV runs out of lines
            break
        current_row = all_rows.pop()
        row_id, lat, long = current_row[0].strip(), current_row[1].strip(), current_row[2].strip()

        coord_pair = (lat, long)

        if coord_pair in existing_coords: #chose a chose a coord pair already chosen this session
            print(f"Skipping {coord_pair} - already exists.")
            continue
        print(f"Processing new line: ID {row_id} at {coord_pair}")

        new_file_path = new_folder_path / f"{lat}_{long}.png"
        if new_file_path.exists():
            # randomly chose a coord pair that was already chosen and recorded before, add it to the hashmap and go to next iteration
            existing_coords.add(coord_pair)
            continue
        sleep(10)
        try:
            ndvi = get_landsat_ndvi_matrix(float(lat), float(long), half_side_meters=random.randint(1400,2100))
            print("Found Data")
        except Exception as e:
            print(f"An error occurred: {e}")
            #something broke, abort
            break
        ndviimage = Image.fromarray(np.clip(np.round(ndvi * 65535.0), 0, 65535).astype(np.uint16))

        ndviimage.save(new_file_path)
        
        # adding to existing hasmap
        existing_coords.add(coord_pair)

Created folder: ndvidata/usEastFarms
Processing new line: ID 1304049312 at ('41.3348003', '-95.8892444')
connecting to API
searching for recent landsat items
Found best match: LC08_L2SP_028031_20260420_02_T1 (Date: 2026-04-20T17:05:31.927610Z)
streaming and cropping Band 4 (Red)
streaming and cropping Band 5 (NIR)
calculating ndvi array
NDVI processing complete. Result shape: (134, 134)
Found Data
Processing new line: ID 881711642 at ('43.9270758', '-92.4603534')
connecting to API
searching for recent landsat items
Found best match: LC08_L2SP_026030_20260524_02_T1 (Date: 2026-05-24T16:52:19.873912Z)
streaming and cropping Band 4 (Red)
streaming and cropping Band 5 (NIR)
calculating ndvi array
NDVI processing complete. Result shape: (134, 134)
Found Data
Processing new line: ID 779054384 at ('41.9830487', '-93.5789373')
connecting to API
searching for recent landsat items
Found best match: LC08_L2SP_027031_20260616_02_T1 (Date: 2026-06-16T16:59:00.643963Z)
streaming and cropping Band 4 